In [6]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=86e627d6aeadc8a6a994549fdf8fc45a54a538cfffede5b9150d755ee535ad6f
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [9]:
!pip install nltk
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_excel('/content/sample_data/Capstone-Dataset.xlsx')

,Query,Expected Output,Response from Agent
0,I am framed for stealing a necklace from a jew...,NaN,"As per the Bharatiya Nyaya Sanhita, 2023 (BNS)..."
1,Complicated Relationship Turned Legal Threat: ...,Pretty bad. You had sexual relationship with a...,"Based on the details you've shared, here's a b..."


In [ ]:
df = df.iloc[1:]
df

In [4]:
expected_texts = df.iloc[:, 1].fillna("").tolist()
actual_texts = df.iloc[:, 2].fillna("").tolist()

# Initialize the TF-IDF Vectorizer.
# Fit on both expected and actual texts to have a common vocabulary.
vectorizer = TfidfVectorizer().fit(expected_texts + actual_texts)

# Function to compute cosine similarity for two texts
def compute_similarity(text1, text2):
    tfidf_text1 = vectorizer.transform([text1])
    tfidf_text2 = vectorizer.transform([text2])
    return cosine_similarity(tfidf_text1, tfidf_text2)[0][0]

# Compute the similarity for each row and store the results in a new column.
similarity_scores = []
for expected, actual in zip(expected_texts, actual_texts):
    score = compute_similarity(expected, actual)
    similarity_scores.append(score)

df['similarity'] = similarity_scores

# Display the similarity scores for each row
print(df[['similarity']])


   similarity
1    0.508552


In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score
# from rouge_score import rouge_scorer
# import Levenshtein
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('wordnet')


# Assuming:
# - Column 2 (index 1) holds the expected responses.
# - Column 3 (index 2) holds the actual responses.
expected_texts = df.iloc[:, 1].fillna("").tolist()
actual_texts = df.iloc[:, 2].fillna("").tolist()

# ---------------------------
# Cosine Similarity with TF-IDF
# ---------------------------
# Fit a TF-IDF vectorizer on all texts to build a common vocabulary.
vectorizer = TfidfVectorizer().fit(expected_texts + actual_texts)

def compute_cosine_similarity(text1, text2):
    tfidf_text1 = vectorizer.transform([text1])
    tfidf_text2 = vectorizer.transform([text2])
    return cosine_similarity(tfidf_text1, tfidf_text2)[0][0]

# ---------------------------
# BLEU Score
# ---------------------------
def compute_bleu(expected, actual):
    # Tokenize texts and lower-case them
    reference = word_tokenize(expected.lower())
    hypothesis = word_tokenize(actual.lower())
    # Using sentence_bleu with a single reference
    return sentence_bleu([reference], hypothesis)

# ---------------------------
# METEOR Score
# ---------------------------
def compute_meteor(expected, actual):
    # Tokenize the input texts before passing to meteor_score
    expected_tokens = word_tokenize(expected)
    actual_tokens = word_tokenize(actual)
    return meteor_score([expected_tokens], actual_tokens)

# ---------------------------
# ROUGE-L Score (F1)
# ---------------------------
# def compute_rouge(expected, actual):
#     scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
#     scores = scorer.score(expected, actual)
#     return scores['rougeL'].fmeasure

# ---------------------------
# Normalized Edit (Levenshtein) Similarity
# # ---------------------------
# def compute_normalized_edit_distance(expected, actual):
#     dist = Levenshtein.distance(expected, actual)
#     max_len = max(len(expected), len(actual))
#     # Avoid division by zero; if both strings are empty, consider them identical.
#     if max_len == 0:
#         return 1.0
#     # Convert distance to a similarity score (1 means identical, 0 means completely different)
#     return 1 - dist / max_len

# ---------------------------
# Jaccard Similarity
# ---------------------------
def compute_jaccard(expected, actual):
    set1 = set(word_tokenize(expected.lower()))
    set2 = set(word_tokenize(actual.lower()))
    union = set1.union(set2)
    if not union:
        return 1.0
    return len(set1.intersection(set2)) / len(union)

# ---------------------------
# Dice Coefficient
# ---------------------------
def compute_dice(expected, actual):
    set1 = set(word_tokenize(expected.lower()))
    set2 = set(word_tokenize(actual.lower()))
    if (len(set1) + len(set2)) == 0:
        return 1.0
    return 2 * len(set1.intersection(set2)) / (len(set1) + len(set2))

# ---------------------------
# Compute All Metrics for Each Row
# ---------------------------
cosine_scores = []
bleu_scores = []
meteor_scores = []
rouge_scores = []
edit_distance_scores = []
jaccard_scores = []
dice_scores = []

for expected, actual in zip(expected_texts, actual_texts):
    cosine_scores.append(compute_cosine_similarity(expected, actual))
    bleu_scores.append(compute_bleu(expected, actual))
    meteor_scores.append(compute_meteor(expected, actual))
    # rouge_scores.append(compute_rouge(expected, actual))
    # edit_distance_scores.append(compute_normalized_edit_distance(expected, actual))
    jaccard_scores.append(compute_jaccard(expected, actual))
    dice_scores.append(compute_dice(expected, actual))

# Add the metrics as new columns to the DataFrame
df['cosine_similarity'] = cosine_scores
df['bleu_score'] = bleu_scores
df['meteor_score'] = meteor_scores
# df['rouge_l'] = rouge_scores
# df['normalized_edit_similarity'] = edit_distance_scores
df['jaccard_similarity'] = jaccard_scores
df['dice_coefficient'] = dice_scores

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


   cosine_similarity    bleu_score  meteor_score  jaccard_similarity  \
1           0.508552  9.551033e-79      0.276019             0.16129   

   dice_coefficient  
1          0.277778  


In [17]:
# Loop over each row in the DataFrame and print metrics with simple descriptive text
for index, row in df.iterrows():
    print(f"Row {index + 1}:")
    print(f"  Cosine Similarity (TF-IDF based similarity): {row['cosine_similarity']}")
    print(f"  BLEU Score (n-gram overlap): {row['bleu_score']}")
    print(f"  METEOR Score (considers synonyms and word order): {row['meteor_score']}")
    # print(f"  ROUGE-L Score (longest common subsequence match): {row['rouge_l']}")
    # print(f"  Normalized Edit Similarity (based on edit distance): {row['normalized_edit_similarity']}")
    print(f"  Jaccard Similarity (set overlap of words): {row['jaccard_similarity']}")
    print(f"  Dice Coefficient (alternative set overlap measure): {row['dice_coefficient']}")
    print("-" * 50)


Row 2:
  Cosine Similarity (TF-IDF based similarity): 0.5085521260347327
  BLEU Score (n-gram overlap): 9.55103320570012e-79
  METEOR Score (considers synonyms and word order): 0.27601876955161625
  Jaccard Similarity (set overlap of words): 0.16129032258064516
  Dice Coefficient (alternative set overlap measure): 0.2777777777777778
--------------------------------------------------
